John S. Anvik, 2025  
University of Lethbridge Student ID: 001224816

### Import Packages

In [30]:
# Import packages

library(geosphere) # For distm(), for calculating distances
library(tidyverse)
# library(dagitty)
# library(brms)
# library(forestplot)
# library(lqmm)

### Load and Trim Dataset

In [31]:
# minizord.csv already removed non-songs, post-dawn, unmeasureable songs, acoustic outliers
# see "Data Preprocessing/01 preprocessing.ipynb"
data <- read_csv("../data/minizord.csv")

# Only look at North data for now
data <- data %>% filter(neighbourhood == "north") %>% ungroup()

Rows: 7594 Columns: 51
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr   (4): ID, IST, neighbourhood, Location_Number
dbl  (45): time_rel_sunrise, UST, D_M1_M2, D_M1_M3, D_M1_M4, D_M1_M5, D_M1_M...
lgl   (1): RepeatST
dttm  (1): DateTime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


### Reformat Data: Row for each possible matching opportunity of focal male

In [32]:
# Get all unique focal male IDs (excluding NAs)
focal_ids <- unique(na.omit(data$ID))
print(focal_ids)

# Initialize an empty list to store the output dataframes
focal_outputs <- list()

# Just focusing on one focal male for now (central male LbOW)
focal_id <- "LbOW"
# Get rows in which ID == focal_id
focals_data <- filter(data, ID == focal_id)

# Get unique set of USTs for the focal (including NA values)
focal_repertoire <- unique(focals_data$UST)

# Find all rows that have a UST in focal's repertoire, not including NA values
focal_ind <- data %>% filter(UST %in% focal_repertoire & !is.na(UST))
# Group these rows by date, create columns to show focal's next UST
focal_ind <- focal_ind %>%
    group_by(as_date(DateTime)) %>% # Group by date so we're not comparing "last song" across days
    mutate(focal_next_UST = if_else(ID == focal_id, UST, NA)) %>%
    fill(focal_next_UST, .direction = "up")

# Create column focal_matches : binary variable (is there a song type match?)
# If the focal bird is the singer, it cannot match itself, return 0
# If the focal bird is not the singer, does the song type match the focal's next song? (within day)
focal_ind <- focal_ind %>%
    mutate(focal_matches = if_else(ID == focal_id, NA,
                                    if_else(focal_next_UST == UST, 1, 0)))

[1] "DgTbO"  "LbOW"   "RRO"    "RbDgY"  "TbTbLg" "YKY"   


In [33]:
# Just focusing on one focal male for now (central male LbOW)
focal_id <- "LbOW"
focal_match_data <- data

In [36]:
# Make column for time since Male N's last song
# NA values at top of each day until that male sings
for (i in 2:6) {
    time_last_song <- sprintf("time_M%i_last_song", i)
    time_since_last_song <- sprintf("time_since_M%i_last_song", i)
    print(time_last_song)
    focal_match_data <- focal_match_data %>% 
    group_by(date(DateTime)) %>%
    mutate("{new_col_name}" := if_else(ID == focal_id, time_rel_sunrise, NA)) %>%
    mutate("{time_since_last_song}" := time_rel_sunrise - !!sym(time_last_song)) %>%
    ungroup() %>%
    fill(new_col_name, .direction = "down")
}

[1] "time_M2_last_song"


ERROR: [1m[33mError[39m in `mutate()`:[22m
[1m[22m[36mℹ[39m In argument: `time_since_M2_last_song = time_rel_sunrise -
  time_M2_last_song`.
[36mℹ[39m In group 1: `date(DateTime) = 2022-05-13`.
[1mCaused by error:[22m
[33m![39m object 'time_M2_last_song' not found


In [29]:
slice_sample(focal_match_data, n=1)

ID,DateTime,time_rel_sunrise,UST,IST,D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,⋯,time_M2_last_song,time_M3_last_song,time_M4_last_song,time_M5_last_song,time_M6_last_song,time_since_M2_last_song,time_since_M3_last_song,time_since_M4_last_song,time_since_M5_last_song,time_since_M6_last_song
<chr>,<dttm>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
TbTbLg,2022-05-14 05:41:11,-888.1193,8,TbTbLg_4,NA,61.66927,67.67739,62.95599,53.6376,⋯,-1386.524,-1386.524,-1386.524,-1386.524,-583.7896,498.4043,498.4043,498.4043,498.4043,NA


In [ ]:
# Select only rows with focal male's songs
focal_match_data <- focal_match_data %>%
    filter(ID == focal_id) %>%
# Input variables: M2-M6's last UST, time since M1-M6 UST, # M2-M6's distance
    select(
        "D_M1_M2", "D_M1_M3", "D_M1_M4", "D_M1_M5", "D_M1_M6",
        "M2_last_UST", "M3_last_UST", "M4_last_UST", "M5_last_UST", "M6_last_UST",
        number_of_matches, does_it_match,
        DateTime, time_rel_sunrise
    )
# Make sure UST is a "levels" variable (categorical)
# If UST not in M1's repertoire, set to NA

# Either that or create other variables: M2_in_rep, M3_in_rep, etc.

In [ ]:



focal_match_data <- focal_match_data %>% 
    group_by(date) %>%
    mutate(time_focals_last_song = if_else(ID == focal_id, time_rel_sunrise, NA)) %>%
    fill(time_focals_last_song, .direction = "down") %>%
    mutate(time_since_focals_last_song = time_rel_sunrise - time_focals_last_song) %>%
    ungroup() %>%
    # make column which contains the time of focal's songs (like "time_focal's_last_song", but not f-filled)
    mutate(just_focal_time = ifelse(ID == focal_id, time_rel_sunrise, NA))

In [ ]:
focal_match_data

D_M1_M2,D_M1_M3,D_M1_M4,D_M1_M5,D_M1_M6,M2_last_UST,M3_last_UST,M4_last_UST,M5_last_UST,M6_last_UST
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NA,46.47336,67.67739,62.95599,53.63760,NA,166,3,10,98
NA,53.50176,67.67739,62.95599,53.63760,NA,35,10,10,98
NA,50.01110,67.67739,62.95599,53.63760,NA,35,22,33,98
NA,46.47336,67.67739,62.95599,53.63760,NA,35,16,10,9
NA,54.16852,67.67739,62.95599,53.63760,NA,167,5,10,13
NA,52.93973,NA,62.95599,91.08596,NA,3,8,33,10
NA,52.93973,62.79695,62.95599,91.08596,NA,166,60,20,13
NA,52.93973,67.67739,62.95599,104.82346,NA,27,62,33,10
NA,52.93973,61.21936,62.95599,53.63760,NA,1,62,33,12


In [ ]:
# Group by date and UST to make column showing time focal last sang a song type
focal_ind <- focal_ind %>%
    group_by(date, UST) %>%
    mutate(time_focals_last_rend_ST = lag(just_focal_time)) %>%
    fill(time_focals_last_rend_ST, .direction = "down") %>%
    mutate(time_since_focals_last_rend_ST = -1 * (time_focals_last_rend_ST - time_rel_sunrise))

In [ ]:
# Select only necessary variables
focal_ind_lean <- focal_ind %>%
    select(index, Focal_ID, date, ID, time_rel_sunrise, UST, number_of_matches,
        does_it_match, focal_matches, dist_to_focal, time_since_focals_last_song,
        time_focals_last_rend_ST, time_since_focals_last_rend_ST, focal_next_UST:time_since_focals_last_rend_ST) %>%
# Remove focal's songs (which he cannot match)
    filter(ID != focal_id) %>%
    ungroup()

In [ ]:
head(focal_ind_lean)

index,Focal_ID,date,ID,time_rel_sunrise,UST,number_of_matches,does_it_match,focal_matches,dist_to_focal,time_since_focals_last_song,time_focals_last_rend_ST,time_since_focals_last_rend_ST,focal_next_UST,time_focals_last_song,just_focal_time
<dbl>,<chr>,<date>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
3,LbOW,2022-05-13,DgTbO,-1909.821,20,0,0,0,NA,NA,NA,NA,10,NA,NA
7,LbOW,2022-05-13,DgTbO,-1871.335,8,1,1,0,NA,NA,NA,NA,10,NA,NA
9,LbOW,2022-05-13,DgTbO,-1854.549,20,0,0,0,NA,NA,NA,NA,10,NA,NA
11,LbOW,2022-05-13,DgTbO,-1842.433,10,0,0,1,NA,NA,NA,NA,10,NA,NA
19,LbOW,2022-05-13,DgTbO,-1770.057,20,0,0,0,NA,NA,NA,NA,10,NA,NA
21,LbOW,2022-05-13,DgTbO,-1755.132,8,0,0,0,NA,NA,NA,NA,10,NA,NA


### Export Data

In [ ]:
write_csv(focal_ind_lean, "../data/matching_opportunities.csv")